In [ ]:
Board=[
    ['R1','H1','B1','Q1','K1','B1','H1','R1'],
    ['p1','p1','p1','p1','p1','p1','p1','p1'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['P0','P0','P0','P0','P0','P0','P0','P0'],
    ['R0','H0','B0','Q0','K0','B0','H0','R0']]

Variants=[]
castlingL0,castlingR0=True,True
castlingL1,castlingR1=True,True

rv = {'R':[[0,1],[1,0],[0,-1],[-1,0],1],
            'B':[[1,1],[-1,-1],[1,-1],[-1,1],1],
            'Q':[[1,1],[-1,-1],[1,-1],[-1,1],[0,1],[1,0],[0,-1],[-1,0],1],
            'H':[[1,2],[2,1],[-1,-2],[-2,-1],[-1,2],[-2,1],[1,-2],[2,-1],0],
            'P':[[-1,-1],[1,-1],0],
            'p':[[-1,1],[1,1],0],
            'K':[[1,1],[-1,-1],[1,-1],[-1,1],[0,1],[1,0],[0,-1],[-1,0],0]
            }

def CheckShah(B_W:str) -> bool: #аргумент B_W принимает значение 0 или 1. 0-Если интересует шах белому королю, 1-если черному
    y=0
    for line in Board:
        x =0
        for B in line:
            if B!='.':
                if B_W != B[-1]:
                    for a in rv[B[0]][:-1]:
                        if rv[B[0]][-1] == 1:
                            pos = [x,y]
                            for i in range(8):
                                pos[0] += a[0]
                                pos[1] += a[1]
                                if pos[0]>7 or pos[0]<0 or pos[1]>7 or pos[1]<0: break
                                if Board[pos[1]][pos[0]] != '.':
                                    if Board[pos[1]][pos[0]] == 'K' + B_W:
                                        return True
                                    else:
                                        break
            x+=1
        y+=1
    return False

def ShowVariants(x,y): #x,y-координаты фигуры для которой нужно определить ходы
    global Variants
    Variants=[] #список вариантов ходов
    B=Board[y][x] #B-фигура для которой нужно определить ходы
    for shift in AttackDict[B[0]][0:-1]:#уже знакомый shift-сдвиг
        pos=[x,y] #а также знакомая позиция фигуры-pos
        for i in range(AttackDict[B[0]][-1]*6+1): #если аттака во всё поле, то цикл повториться 7 раз, иначе-1 раз.
            pos[0]+=shift[0]
            pos[1]+=shift[1]#опять смещаем позицию с помощью shift
            if pos[0]>7 or pos[0]<0 or pos[1]>7 or pos[1]<0: break #если X или Y рассматриваемой позиции выходит за пределы поля, то остановить проверку этого направления
            if Board[pos[1]][pos[0]]!='.': #если клетка не пуста
                if Board[pos[1]][pos[0]][1]!=Board[y][x][1]: Variants.append([pos[0],pos[1]]); break #если клетку занимает вражеская фигура то добавить её как вариант хода и остановить проверку этого направления
                else: break #если же клетку заняла дружеская фигура, то остановить эту линию ходов
            elif B[0]!='p' and B[0]!='P': #если клетка пуста, а рассматриваемая фигура не пешка то добавить её как вариант хода. (Пешка не может атаковать пустую клетку)
                Variants.append([pos[0],pos[1]])
    
    if B[0]=='P': #если рассматриваемая фигура-белая пешка то добавим стандартные ходы пешек, производимые без взятия
        pos=[x,y]
        for i in range((y==6)+1): #если пешка на 6-ой линии то цикл повториться 2 раза, иначе-1. (По правилам пешки могут ходить на две клетки если они на "родной линии")
            pos[1]-=1
            if pos[1]<0: break #если вышли за пределы-стоп
            if Board[pos[1]][pos[0]]!='.':break #если клетка впереди не пуста а занята-тоже стоп
            Variants.append([pos[0],pos[1]])#если цикл ещё не остановлен то добавим рассматриваемую клетку как вариант хода

    if B[0]=='p':#все тоже самое для черной пешки
        pos=[x,y]
        for i in range((y==1)+1):
            pos[1]+=1
            if pos[1]>7: break
            if Board[pos[1]][pos[0]]!='.':break
            Variants.append([pos[0],pos[1]])
    
    #Теперь дело за малым-откинуть все ходы, которые ставят своего короля под шах
    
    ForDeletion=[] #список вариантов на удаление
    Board[y][x]='.' #временно уберем рассматриваемую фигуру со стола
    for V in Variants: #переберем все варианты
        remember=Board[V[1]][V[0]] #запоминаем клетку, на которую сейчас поставим фигуру
        Board[V[1]][V[0]]=B #ставим фигуру на это место
        if CheckShah(B[1]): ForDeletion.append(V) #если король под шахом-добавим этот вариант в список на удаление
        Board[V[1]][V[0]]=remember #возвращаем клетку которую запомнили
    Board[y][x]=B #вернём рассматриваемую фигуру на стол
    for Del in ForDeletion: #удалим все недопустимые варианты
        Variants.remove(Del)
    
    if Board[y][x]=='K0': #если рассматриваем ходы для белого короля
        global castlingL0, castlingR0
        if Board[7][0:5]==['R0','.','.','.','K0'] and castlingL0: #если между левой ладьёй и королём пусто, а рокировка с левой ладьёй не запрещена
            Board[7][2],Board[7][3]='K0','K0' #временно поставим два короля в клетки через которые пройдёт король
            if CheckShah('0')==0: #если эти короли не получают шаха, то это значит, что все условия для рокировки есть и можно добавлять ход-рокировку
                Variants.append([2,7])
            Board[7][2],Board[7][3]='.','.' #уберём временных королей
        
        if Board[7][4:8]==['K0','.','.','R0'] and castlingR0: #все тоже самое для рокировки с правой ладьёй
            Board[7][5],Board[7][6]='K0','K0'
            if CheckShah('0')==0:
                Variants.append([6,7])
            Board[7][5],Board[7][6]='.','.'
    if Board[y][x]=='K1':
        global castlingL1, castlingR1
        if Board[0][0:5]==['R1','.','.','.','K1'] and castlingL1:
            Board[0][2],Board[0][3]='K1','K1'
            if CheckShah('1')==0:
                Variants.append([2,0])
            Board[0][2],Board[0][3]='.','.'
        
        if Board[0][4:8]==['K1','.','.','R1'] and castlingR1:
            Board[0][5],Board[0][6]='K1','K1'
            if CheckShah('1')==0:
                Variants.append([6,0])
            Board[0][5],Board[0][6]='.','.'
    

def CheckCheckMate(B_W): #аргумент B_W-как обычно, 0-интерисует мат/пат белых, 1-черных
    global Variants
    y=0
    for Brd in Board: #проверка каждой строки
        x=0
        for B in Brd: #проверка каждого элемента строки
            if B[-1]==B_W: #если найдена фигура нужного цвета то проверить, есть ли для неё хоть один вариант хода. Если да-вернуть 0-мата/пата нет
                ShowVariants(x,y)
                if len(Variants)>0:Variants=[];return 0
            x+=1
        y+=1
    #если дошли до этой строки, то это значит, что ни одна фигура нужного цвета не может сделать ход. Это значит что поставлен мат или пат
    if CheckShah(B_W): Variants=[];return 1 #король под шахом-значит мат, возвращаем 1
    else: Variants=[];return 2 #король не под шахом-пат возвращаем 2
    #обратите внимание что перед тем как вернуть значение, необходимо очистить список Variants, чтобы избежать багов


In [ ]:
a=1
def f():
    a=4

f()
print(a)